In [1]:
import h5py
import halotools
import os
import copy
import warnings
import numpy as np
import math
from astropy.modeling import Fittable1DModel, Parameter,models,fitting
import pickle
import warnings
import numpy.ma as ma
import pandas as pd
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from scipy.stats import spearmanr,kendalltau,pearsonr
import palettable
import deepdish as dd
import matplotlib.cm as cm

from matplotlib.collections import LineCollection
from photutils.isophote import EllipseGeometry
from photutils.aperture import EllipticalAperture,EllipticalAnnulus
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from matplotlib import colors
from astroML.stats import binned_statistic_2d
from matplotlib_venn import venn2, venn3
from scipy.ndimage.filters import gaussian_filter

from astropy.utils.console import ProgressBar
from astropy.table import QTable,hstack
plt.rc('text', usetex=True)
rcParams.update({'axes.linewidth': 1.5})
rcParams.update({'xtick.direction': 'in'})
rcParams.update({'ytick.direction': 'in'})
rcParams.update({'xtick.minor.visible': 'True'})
rcParams.update({'ytick.minor.visible': 'True'})
rcParams.update({'xtick.major.pad': '7.0'})
rcParams.update({'xtick.major.size': '8.0'})
rcParams.update({'xtick.major.width': '1.5'})
rcParams.update({'xtick.minor.pad': '7.0'})
rcParams.update({'xtick.minor.size': '4.0'})
rcParams.update({'xtick.minor.width': '1.5'})
rcParams.update({'ytick.major.pad': '7.0'})
rcParams.update({'ytick.major.size': '8.0'})
rcParams.update({'ytick.major.width': '1.5'})
rcParams.update({'ytick.minor.pad': '7.0'})
rcParams.update({'ytick.minor.size': '4.0'})
rcParams.update({'ytick.minor.width': '1.5'})
rcParams.update({'axes.titlepad': '10.0'})
rcParams.update({'font.size': 35})

In [2]:
fig_dir="/Users/xushuo/work/Papers/CenvsSat/Figure/"
data_dir='/Users/xushuo/work/Submit/CenvsSat/Data/'

In [3]:
with open(data_dir+"galaxies_tng300_072_hmc_13_aperture_mass.txt",'rb') as f: 
    tab=pickle.load(f)
mask=(tab['proj']=='xy')
tab1=tab[mask]

In [4]:
with open(data_dir+"galaxies_tng300_072_hmc_13_richness_satellite_mass.txt",'rb') as f: 
    tab_rich0=pickle.load(f)

In [5]:
with open(data_dir+"galaxies_tng300_072_hmc_13_cylinder_richness_satellite_mass.txt",'rb') as f: 
    tab_rich=pickle.load(f)

In [6]:
tab2=hstack([tab[mask],tab_rich0[tab_rich0.colnames[2:]],tab_rich[tab_rich.colnames[2:]]])

In [7]:
with open(data_dir+"galaxies_tng300_072_mah_hmc.txt",'rb') as f: 
    tab_mah=pickle.load(f)

In [8]:
df = pd.read_csv(data_dir+'tng_snap.csv')
age_tab=np.asarray(df['age'])
z_tab=np.asarray(df['redshift'])

In [9]:
with open(data_dir+'galaxies_tng300_072_hmc_lensing_prof.txt','rb') as f:
    lensing_cat=pickle.load(f)

In [10]:
cmap1=palettable.lightbartlein.diverging.BlueDarkRed18_10.mpl_colormap

In [11]:
def topNselect(data,name,number):
    data_meta=data.group_by(name)
    data0=data_meta[-number::]
    return data0
def topNoutselect(data,name1,name2,number):
    data_meta=data.group_by(np.log10(data[name1]-data[name2]))
    data0=data_meta[-number::]
    return data0
def linear1d(x,a,b):
    return a*x+b
def mpeakfind(tab,bin_num=30):
    numb,bins=np.histogram(tab,bins=bin_num)
    max_value=max(numb)
    max_index=list(numb).index(max_value)
    mpeak=bins[max_index]
    return mpeak
def resample_list(tab):
    length=len(tab)
    list0=np.random.randint(length,size=length)
    return tab[list0]
def bootstrap_lensing(gal_num_list,rp_bins,times=5000):
    lens_list=[]
    for ii in range(times):
        gal_rand=resample_list(gal_num_list)
        lens_tab=[]
        for i in gal_rand:
            lens_tab.append(lensing_cat[i])
        lens=np.mean(lens_tab,axis=0)
        lens_list.append(lens)
    upper=[]
    lower=[]
    for j in range(len(rp_bins)-1):
        upper.append(np.percentile(np.asarray(lens_list).T[j],84))
        lower.append(np.percentile(np.asarray(lens_list).T[j],16))
    return np.asarray(upper),np.asarray(lower)

In [12]:
logrp_bins = np.linspace(-1,1.5,20)
rp_bins = 10**logrp_bins
logrp_mids = 0.5*(logrp_bins[:-1] + logrp_bins[1:])

In [13]:
10**logrp_mids

In [14]:
mpeak_hist=[[],]
gal_num_exl=[]
for gal_num in range(3388):
    if (len(tab_mah[gal_num][0])>0):
        xdata=np.flip(np.append([72],tab_mah[gal_num][0].astype(np.int64)))
        ydata=np.flip(np.append([tab2['mass_halo'][gal_num]],tab_mah[gal_num][1]*1e10/0.6774))
        xdata0=[xdata[0],]
        ydata0=[ydata[0],]
        for i in range(1,len(xdata)):
            xdata0.append(xdata[i])
            if ydata[i]>ydata0[i-1]:
                ydata0.append(ydata[i])
            else:
                ydata0.append(ydata0[i-1])
    else:
        gal_num_exl.append(gal_num)
        xdata0=[None]
        ydata0=[None]
    mpeak_hist.append([xdata0,ydata0])
del mpeak_hist[0]

In [15]:
mask_exl=(tab2['aper_100_gal']>0)
for i in gal_num_exl:
    mask_exl=mask_exl&(tab2['gal_num']!=i)

In [16]:
mask1=(tab2['c_200c']<15)&mask_exl

In [17]:
age_tab[mpeak_hist[0][0]]

In [18]:
age_list=age_tab[32:73]*0.9999

In [19]:
def stack_mah(gal_num_list):
    prof_curve=[]
    median=[]
    upper=[]
    lower=[]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for gal_num in gal_num_list:
            gal_num=int(gal_num)
            x=age_tab[mpeak_hist[gal_num][0]]
            y=np.asarray(mpeak_hist[gal_num][1])
            mask00=(x>1)
            if np.min(x[mask00])<3:
                f=interp1d(x[mask00],y[mask00]/y[mask00][-1],kind='cubic')
                prof_curve.append(f(age_list))
        for i in range(len(age_list)):
            median.append(np.median(np.asarray(prof_curve).T[i]))
            upper.append(np.percentile(np.asarray(prof_curve).T[i],84))
            lower.append(np.percentile(np.asarray(prof_curve).T[i],16))
    return median,upper,lower

In [20]:
def stack_mah_mean(gal_num_list):
    prof_curve=[]
    median=[]
    upper=[]
    lower=[]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for gal_num in gal_num_list:
            gal_num=int(gal_num)
            x=age_tab[mpeak_hist[gal_num][0]]
            y=np.asarray(mpeak_hist[gal_num][1])
            mask00=(x>1)
            if np.min(x[mask00])<3:
                f=interp1d(x[mask00],y[mask00]/y[mask00][-1],kind='cubic')
                prof_curve.append(f(age_list))
        for i in range(len(age_list)):
            median.append(np.median(np.asarray(prof_curve).T[i]))
    return np.asarray(median)

In [21]:
def bootstrap_mah(gal_num_list,age_list,times=5000):
    mah_list=[]
    for ii in range(times):
        gal_rand=resample_list(gal_num_list)
        mah=stack_mah_mean(gal_rand)
        mah_list.append(mah)
    upper=[]
    lower=[]
    for j in range(len(age_list)):
        upper.append(np.percentile(np.asarray(mah_list).T[j],84))
        lower.append(np.percentile(np.asarray(mah_list).T[j],16))
    return np.asarray(upper),np.asarray(lower)

In [22]:
def bootstrap_ratio_mah(gal_num_upp,gal_num_down,age_list,times=5000):
    ratio_list=[]
    for ii in range(times):
        gal_rand_upp=resample_list(gal_num_upp)
        mah_upp=stack_mah_mean(gal_rand_upp)
        gal_rand_down=resample_list(gal_num_down)
        mah_down=stack_mah_mean(gal_rand_down)
        ratio_list.append(mah_down/mah_upp)
    upper=[]
    lower=[]
    for j in range(len(age_list)):
        upper.append(np.percentile(np.asarray(ratio_list).T[j],84))
        lower.append(np.percentile(np.asarray(ratio_list).T[j],16))
    return np.asarray(upper),np.asarray(lower)
def bootstrap_ratio_lensing(gal_num_upp,gal_num_down,rp_bins,times=5000):
    ratio_list=[]
    for ii in range(times):
        gal_rand_upp=resample_list(gal_num_upp)
        lens_tab=[]
        for i in gal_rand_upp:
            lens_tab.append(lensing_cat[i])
        lens_upp=np.mean(lens_tab,axis=0)
        gal_rand_down=resample_list(gal_num_down)
        lens_tab=[]
        for i in gal_rand_down:
            lens_tab.append(lensing_cat[i])
        lens_down=np.mean(lens_tab,axis=0)
        ratio_list.append(lens_down/lens_upp)
    upper=[]
    lower=[]
    for j in range(len(rp_bins)-1):
        upper.append(np.percentile(np.asarray(ratio_list).T[j],84))
        lower.append(np.percentile(np.asarray(ratio_list).T[j],16))
    return np.asarray(upper),np.asarray(lower)

In [23]:
age_tab[[33,50,59,72]]

In [24]:
name='richness_cut_r200_10'
label=r'$\log\lambda_{\log M_\star>10,R_{200}}$'
top_numb=200
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [26]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.5,14.7,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2


plt.savefig(fig_dir+"FigA1.pdf",dpi=150,bbox_inches='tight')

In [27]:
name='richness_cut_r200_10'
label=r'$\log\lambda_{\log M_\star>10,R_{200}}$'
top_numb=400
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [31]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.3,14.3,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2


plt.savefig(fig_dir+"FigA2.pdf",dpi=150,bbox_inches='tight')

In [32]:
name='richness_cut_r200_10'
label=r'$\log\lambda_{\log M_\star>10,R_{200}}$'
top_numb=800
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [33]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.1,14.1,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2


plt.savefig(fig_dir+"FigA3.pdf",dpi=150,bbox_inches='tight')

In [36]:
name='richness_cut_r200_10_Q'
label=r'$\log\lambda_{\log M_\star>10,R_{200},Q}$'
top_numb=800
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [37]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.1,14.1,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2

In [39]:
name='richness_cut_r200_10_xy_40_Q'
label=r'$\log\lambda_{\log M_\star>10,xy,40,Q}$'
top_numb=800
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [40]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.1,14.1,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2

In [41]:
name='richness_cut_r200_10_xy_40'
label=r'$\log\lambda_{\log M_\star>10,xy,40}$'
top_numb=800
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

median_upp=stack_mah_mean(up_data_gal_num)
upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
median_down=stack_mah_mean(down_data_gal_num)
upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)



In [42]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,18))
gs_left = GridSpec(2, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(2, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
gs_right = GridSpec(2, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2))  # No vertical space
ax1 = fig.add_subplot(gs_left[0])
ax3 = fig.add_subplot(gs_left[1])

ax2 = fig.add_subplot(gs_middle[0])
ax2_r = fig.add_subplot(gs_middle[1])

ax20 = fig.add_subplot(gs_right[0])
ax20_r = fig.add_subplot(gs_right[1])

ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
            facecolors='none', edgecolors='darkgrey',alpha=0.5)
ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
            np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
            c='grey',alpha=0.7)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
ax1.set_xlabel(label, fontsize=50)
ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
ax2_r.set_xscale('log')
ax2_r.set_xlim(0.12,15)

ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax20.set_xlim(3.1,9.4)
ax20.set_ylim(0.05,1.03)
ax21=ax20.twiny()
x1,x2=ax20.get_xlim()
ax21.set_xlim(x1,x2)
ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax21.set_xlabel(r'$z$', fontsize=50)
ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
binlist=np.arange(13.1,14.1,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
ax20_r.set_xlim(3.1,9.4)



ax11 = fig.add_axes([0.065, 0.55, 0.14, 0.14])

venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
              len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
              len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


# Hide set labels (labels below Venn diagram)
venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
venn.get_label_by_id('B').set_text('')  # Remove label for Set 2

In [32]:
name='aper_30_gal'
label=r'$\log[M_{\star,30}/M_\odot]$'
top_numb=500
ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
xdata=np.log10(tab2[mask1][name])
mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
index_list=np.where(mask_h2==True)[0]
true_num=top_numb-len(xdata[mask_h1])
array=np.zeros(len(index_list))
array[:true_num]=1
np.random.shuffle(array)
for j in range(len(index_list)):
    index=index_list[j]
    mask_h2[index]=mask_h2[index]&bool(array[j])
mask_h=mask_h1|mask_h2
mask_up=mask_l&(~mask_h)
mask_down=mask_h&(~mask_l)
mask_cro=mask_h&mask_l
    
up_data=tab2[mask1][mask_up]
down_data=tab2[mask1][mask_down]
up_data_gal_num=up_data['gal_num']
down_data_gal_num=down_data['gal_num']
up_data_lens_tab=[]
for i in up_data_gal_num:
    up_data_lens_tab.append(lensing_cat[i])
up_data_lens=np.mean(up_data_lens_tab,axis=0)
up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
down_data_lens_tab=[]
for i in down_data_gal_num:
    down_data_lens_tab.append(lensing_cat[i])
down_data_lens=np.mean(down_data_lens_tab,axis=0)
down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)

fig=plt.figure(figsize=(30,20))
gs = fig.add_gridspec(2,2, hspace=0, wspace=0.15,height_ratios=(0.8,0.2))
axs=gs.subplots(sharex='col')
ax2=axs[0,0]
ax1 = ax2.inset_axes([0.11, 0.1, 0.25,0.25], transform=ax2.transAxes)
ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)][name]),
            np.log10(tab2[mask1][(~mask_up)&(~mask_down)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)]['aper_50_gal']),
            c='darkgrey',alpha=0.5)
ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$')
ax1.set_xlabel(label)
ax1.text(0.8,0.1,r'\rm (a1)',transform=ax1.transAxes,fontsize=35)
ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
__=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
__=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
ax2.set_xscale('log')
ax2.set_xlim(0.12,15)
ax2.legend()
__=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[h M_{\odot} / {\rm pc}^2]$', fontsize=50)
ax2_r=axs[1,0]
ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)
ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
__=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
__=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
ax2_r.set_ylim(0.5,2.2)
ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')


ax2=axs[0,1]
median_upp=stack_mah_mean(up_data_gal_num)
upper,lower=bootstrap_mah(up_data_gal_num,age_list)
ax2.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
ax2.fill_between(age_list,lower,upper,color='orangered',lw=3,alpha=0.3)
median_down=stack_mah_mean(down_data_gal_num)
upper,lower=bootstrap_mah(down_data_gal_num,age_list)
ax2.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
ax2.fill_between(age_list,lower,upper,color='steelblue',lw=3,alpha=0.3)

ax2.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
ax2.set_xlim(3.1,9.4)
ax20=ax2.twiny()
x1,x2=ax2.get_xlim()
ax20.set_xlim(x1,x2)
ax20.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
ax20.set_xlabel(r'$z$')
ax2_r=axs[1,1]
ax2_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
ax2_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
upper,lower=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)
ax2_r.fill_between(age_list,upper,lower,color='steelblue',alpha=0.3)
ax2_r.plot(age_list,1+0*age_list,ls='--',color='black')
__=ax2_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
median,upper,lower=stack_mah(up_data_gal_num)
ax2.fill_between(age_list,lower,upper,color='red',alpha=0.1)
median,upper,lower=stack_mah(down_data_gal_num)
ax2.fill_between(age_list,lower,upper,color='blue',alpha=0.1)
ax3 = fig.add_axes([0.54,0.7,0.12,0.18],transform=ax2.transAxes)
ax4 = fig.add_axes([0.77,0.33,0.12,0.18],transform=ax2.transAxes)
ax3.text(0.8,0.8,r'\rm (b1)',transform=ax3.transAxes,fontsize=35)
ax4.text(0.8,0.8,r'\rm (b2)',transform=ax4.transAxes,fontsize=35)
binlist=np.arange(13.2,14.3,1/15)
ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$')
binlist=np.arange(1,10,2/3)
ax4.hist(up_data['c_200c'],histtype='step',color='orangered',bins=binlist,lw=2)
ax4.hist(down_data['c_200c'],histtype='step',color='steelblue',bins=binlist,lw=2)
ax4.set_xlabel(r'$c_{200c}$')
ax3.tick_params(left=False,labelleft=False,labelright=True,right=True)

In [33]:
len(up_data)

In [33]:
from matplotlib.gridspec import GridSpec
fig=plt.figure(figsize=(30,54))
gs_left = GridSpec(6, 1, left=0, right=0.19, hspace=0.3)  # More vertical space
gs_middle = GridSpec(6, 1, left=0.255, right=0.61, hspace=0,height_ratios=(0.8,0.2,0.8,0.2,0.8,0.2))  # No vertical space
gs_right = GridSpec(6, 1, left=0.665, right=1, hspace=0,height_ratios=(0.8,0.2,0.8,0.2,0.8,0.2))  # No vertical space

venn_loc_list=[0.755,0.485,0.22]
for k in range(3):
    top_numb=2**(k+1)*100
    name='richness_cut_r200_10'
    label=r'$\log\lambda_{\log M_\star>10,R_{200}}$'
    ydata=np.log10(tab2[mask1]['aper_100_gal']-tab2[mask1]['aper_50_gal'])
    topNout=topNoutselect(tab2[mask1],'aper_100_gal','aper_50_gal',top_numb)
    mask_l=(ydata>=np.min(np.log10(topNout['aper_100_gal']-topNout['aper_50_gal'])))
    xdata=np.log10(tab2[mask1][name])
    mask_h1=(xdata>np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
    mask_h2=(xdata==np.min(np.log10(topNselect(tab2[mask1],name,top_numb)[name])))
    index_list=np.where(mask_h2==True)[0]
    true_num=top_numb-len(xdata[mask_h1])
    array=np.zeros(len(index_list))
    array[:true_num]=1
    np.random.shuffle(array)
    for j in range(len(index_list)):
        index=index_list[j]
        mask_h2[index]=mask_h2[index]&bool(array[j])
    mask_h=mask_h1|mask_h2
    mask_up=mask_l&(~mask_h)
    mask_down=mask_h&(~mask_l)
    mask_cro=mask_h&mask_l
        
    up_data=tab2[mask1][mask_up]
    down_data=tab2[mask1][mask_down]
    up_data_gal_num=up_data['gal_num']
    down_data_gal_num=down_data['gal_num']
    up_data_lens_tab=[]
    for i in up_data_gal_num:
        up_data_lens_tab.append(lensing_cat[i])
    up_data_lens=np.mean(up_data_lens_tab,axis=0)
    up_upp,up_low=bootstrap_lensing(up_data_gal_num,rp_bins)
    down_data_lens_tab=[]
    for i in down_data_gal_num:
        down_data_lens_tab.append(lensing_cat[i])
    down_data_lens=np.mean(down_data_lens_tab,axis=0)
    down_upp,down_low=bootstrap_lensing(down_data_gal_num,rp_bins)
    upper,lower=bootstrap_ratio_lensing(up_data_gal_num,down_data_gal_num,rp_bins)

    median_upp=stack_mah_mean(up_data_gal_num)
    upper_upp,lower_upp=bootstrap_mah(up_data_gal_num,age_list)
    median_down=stack_mah_mean(down_data_gal_num)
    upper_down,lower_down=bootstrap_mah(down_data_gal_num,age_list)
    upper_r,lower_r=bootstrap_ratio_mah(up_data_gal_num,down_data_gal_num,age_list)

    median_0_up,upper_0_up,lower_0_up=stack_mah(up_data_gal_num)
    median_0_down,upper_0_down,lower_0_down=stack_mah(down_data_gal_num)


    ax1 = fig.add_subplot(gs_left[0+2*k])
    ax3 = fig.add_subplot(gs_left[1+2*k])

    ax2 = fig.add_subplot(gs_middle[0+2*k])
    ax2_r = fig.add_subplot(gs_middle[1+2*k])

    ax20 = fig.add_subplot(gs_right[0+2*k])
    ax20_r = fig.add_subplot(gs_right[1+2*k])

    ax1.scatter(np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)][name]),
                np.log10(tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_100_gal']-tab2[mask1][(~mask_up)&(~mask_down)&(~mask_cro)]['aper_50_gal']),
                facecolors='none', edgecolors='darkgrey',alpha=0.5)
    ax1.scatter(np.log10(tab2[mask1][mask_cro][name]),
                np.log10(tab2[mask1][mask_cro]['aper_100_gal']-tab2[mask1][mask_cro]['aper_50_gal']),
                c='grey',alpha=0.7)
    ax1.scatter(np.log10(up_data[name]),np.log10(up_data['aper_100_gal']-up_data['aper_50_gal']),c='orangered',alpha=0.5)
    ax1.scatter(np.log10(down_data[name]),np.log10(down_data['aper_100_gal']-down_data['aper_50_gal']),c='steelblue',alpha=0.5)

    ax1.set_ylabel(r'$\log[M_{\star,[50,100]}/M_\odot]$', fontsize=50)
    ax1.set_xlabel(label, fontsize=50)
    ax1.text(0.08,0.9,r'\rm (a)',transform=ax1.transAxes,fontsize=35)
    ax2.text(0.05,0.9,r'\rm Top %d'%top_numb,transform=ax2.transAxes,fontsize=50)
    __=ax2.plot(10**logrp_mids, 10**logrp_mids*up_data_lens/1e12,color='orangered',label=r'$M_\star$ {\rm selected}',lw=3)
    __=ax2.plot(10**logrp_mids, 10**logrp_mids*down_data_lens/1e12,color='steelblue',label=r'$\lambda$ {\rm selected}',lw=3)
    __=ax2.fill_between(10**logrp_mids, 10**logrp_mids*up_upp/1e12,10**logrp_mids*up_low/1e12,color='orangered',alpha=0.3)
    __=ax2.fill_between(10**logrp_mids, 10**logrp_mids*down_upp/1e12,10**logrp_mids*down_low/1e12,color='steelblue',alpha=0.3)
    ax2.set_xscale('log')
    ax2.set_xlim(0.12,15)
    ax2.legend()
    __=ax2.set_ylabel(r'$R_{\rm p}\times\Delta\Sigma(R_{\rm p})$  $[M_{\odot}{\rm Mpc} / {\rm pc}^2]$', fontsize=50)
    ax2_r.plot(10**logrp_mids,down_data_lens/up_data_lens,color='steelblue',lw=3)
    ax2_r.fill_between(10**logrp_mids,upper,lower,color='steelblue',alpha=0.3)
    __=ax2_r.set_xlabel(r'$R_{\rm p} $  $\rm{[Mpc / h]}$', fontsize=50)
    __=ax2_r.set_ylabel(r'$\Delta\Sigma_{\lambda}/\Delta\Sigma_{M_\star}$',fontsize=50)
    ax2_r.set_ylim(0.5,2.2)
    ax2_r.plot(10**logrp_mids,1+0*logrp_mids,ls='--',color='black')
    ax2_r.set_xscale('log')
    ax2_r.set_xlim(0.12,15)

    ax20.plot(age_list,median_upp,color='orangered',lw=3,alpha=1)
    ax20.fill_between(age_list,lower_upp,upper_upp,color='orangered',lw=3,alpha=0.3)
    ax20.plot(age_list,median_down,color='steelblue',lw=3,alpha=1)
    ax20.fill_between(age_list,lower_down,upper_down,color='steelblue',lw=3,alpha=0.3)

    ax20.set_ylabel(r'$M_{\rm peak}/M_{{\rm peak},z=0.4}$', fontsize=50)
    ax20.set_xlim(3.1,9.4)
    ax20.set_ylim(0.05,1.03)
    if k==0:
        ax21=ax20.twiny()
        x1,x2=ax20.get_xlim()
        ax21.set_xlim(x1,x2)
        ax21.set_xticks(age_tab[[33,50,59,72]],[r'$2.0$',r'$1.0$',r'$0.7$',r'$0.4$'])
        ax21.set_xlabel(r'$z$', fontsize=50)
    ax20_r.set_xlabel(r'\rm Age[Gyr]', fontsize=50)
    ax20_r.plot(age_list,median_down/median_upp,color='steelblue',lw=3)
    ax20_r.fill_between(age_list,upper_r,lower_r,color='steelblue',alpha=0.3)
    ax20_r.plot(age_list,1+0*age_list,ls='--',color='black')
    __=ax20_r.set_ylabel(r'$M_{\rm peak,\lambda}/M_{\rm peak,M_\star}$',fontsize=50)
    ax20.fill_between(age_list,lower_0_up,upper_0_up,color='red',alpha=0.1)
    ax20.fill_between(age_list,lower_0_down,upper_0_down,color='blue',alpha=0.1)
    ax3.text(0.9,0.9,r'\rm (b)',transform=ax3.transAxes,fontsize=35)
    min0=np.min(np.log10(up_data['mass_halo']))
    max0=np.max(np.log10(down_data['mass_halo']))
    binlist=np.arange(min0-0.05,max0+0.05,1/15)
    ax3.hist(np.log10(up_data['mass_halo']),histtype='step',bins=binlist,color='orangered',lw=2)
    ax3.hist(np.log10(down_data['mass_halo']),histtype='step',bins=binlist,color='steelblue',lw=2)
    ax3.set_xlabel(r'$\log[M_{\rm halo}/M_\odot]$', fontsize=50)
    ax20_r.set_xlim(3.1,9.4)



    ax11 = fig.add_axes([0.08, venn_loc_list[k], 0.1, 0.1])

    venn=venn2(subsets=(len(set(tab2[mask1][mask_h]['gal_num'])-set(tab2[mask1][mask_l]['gal_num'])),
                len(set(tab2[mask1][mask_l]['gal_num'])-set(tab2[mask1][mask_h]['gal_num'])),
                len(set(tab2[mask1][mask_h]['gal_num'])&set(tab2[mask1][mask_l]['gal_num']))),ax=ax11)


    venn.get_patch_by_id('10').set_color('orangered')  # Set color for only Set1
    venn.get_patch_by_id('01').set_color('steelblue')  # Set color for only Set2
    venn.get_patch_by_id('11').set_color('grey')  # Set color for the intersection


    # Hide set labels (labels below Venn diagram)
    venn.get_label_by_id('A').set_text('')  # Remove label for Set 1
    venn.get_label_by_id('B').set_text('')  # Remove label for Set 2


plt.savefig(fig_dir+"FigA1.pdf",dpi=150,bbox_inches='tight')